In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## Load Dependencies

In [ ]:
# !pip install obonet
# !pip install biopython
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import obonet, networkx
from Bio import SeqIO

## Load Train Data

### Load Ontology File

In [ ]:
ont_file_path = "/kaggle/input/cafa-6-protein-function-prediction/Train/go-basic.obo"
ont_graph = obonet.read_obo(ont_file_path) # ignore_obsolete: bool = True
reversed_ont_graph = ont_graph.reverse(copy=False)

roots = [node for node in reversed_ont_graph.nodes if not list(reversed_ont_graph.predecessors(node))]
print(f"root nodes: {roots}")

depths = {}
for root in roots:
    # Calculate shortest path lengths from the current root
    lengths = networkx.shortest_path_length(reversed_ont_graph, source=root)
    for term_id, length in lengths.items():
        # Update depth if this path is shorter than a previously found path
        if term_id not in depths or length < depths[term_id]:
            depths[term_id] = length

### Load Train Terms File

In [ ]:
# Load training annotations (protein GO term associations)
train_terms_path = "/kaggle/input/cafa-6-protein-function-prediction/Train/train_terms.tsv"
train_terms_df = pd.read_csv(train_terms_path, sep="\t", header=0)

print("Total annotations (protein-term pairs):", len(train_terms_df))
print("Unique proteins in training:", train_terms_df["EntryID"].nunique())
print("Unique GO terms in training:", train_terms_df["term"].nunique())
display(train_terms_df.head(5))
print("Unique GO terms per aspect:")
print(train_terms_df.groupby("aspect")["term"].nunique())
print("\nTotal annotation count per aspect:")
print(train_terms_df["aspect"].value_counts())

### Load Train Sequences

In [ ]:
train_fasta_path = "/kaggle/input/cafa-6-protein-function-prediction/Train/train_sequences.fasta"

with open(train_fasta_path, "r") as train_fasta_file:
    train_fasta_sequences = list(SeqIO.parse(train_fasta_file,'fasta'))

## Load Test Data

### Load Test Sequences

In [ ]:
test_fasta_path = "/kaggle/input/cafa-6-protein-function-prediction/Test/testsuperset.fasta"

with open(test_fasta_path, "r") as test_fasta_file:
    test_fasta_sequences = list(SeqIO.parse(test_fasta_file,'fasta'))

## Analyze Label Propagation

### Verify Propagation and Propagate If Needed

In [ ]:
# Check hierarchy consistency of training annotations
propagation_violations = 0
for prot, group in train_terms_df.groupby("EntryID"):
    terms = set(group["term"])
    for t in list(terms):
        # check all ancestors of t
        anc = set()
        stack = [t]
        while stack:
            cur = stack.pop()
            for p in list(reversed_ont_graph.predecessors(cur)):
                if p not in anc:
                    anc.add(p)
                    stack.append(p)
        # subtract the term itself
        anc.discard(t)
        # Now anc contains all ancestor terms of t
        missing = anc - terms
        if missing:
            propagation_violations += 1
            break  # no need to find more for this protein

print("Proteins with a child term but missing an ancestor term:", propagation_violations)
print(f"Total Number of Proteins: {train_terms_df['EntryID'].nunique()}")

In [ ]:
# Function to propagate a set of GO terms to include all ancestors
def propagate_terms(term_set):
    expanded = term_set
    stack = list(term_set)
    while stack:
        cur = stack.pop()
        for p in list(reversed_ont_graph.predecessors(cur)):
            if p not in expanded:
                expanded.add(p)
                stack.append(p)
    return expanded

propagated_rows = []
for prot, group in train_terms_df.groupby("EntryID"):
    prop_terms = propagate_terms(set(group['term']))
    propagated_rows.append(
        {
            "EntryID": prot,
            "term": list(prop_terms)
        }
    )

In [ ]:
train_terms_df_propagated = pd.DataFrame(propagated_rows).explode("term")
term_ns_map = {k: v['namespace'] for k,v in ont_graph.nodes(data=True)}
train_terms_df_propagated['aspect'] = train_terms_df_propagated['term']\
    .map(term_ns_map)\
    .map(
        {
            'biological_process': "P",
            'cellular_component': "C",
            'molecular_function': "F"
        }
    )

print(train_terms_df_propagated.shape)
print(f"aspects per records distribution: {train_terms_df_propagated['aspect'].map(len).value_counts()}")
display(train_terms_df_propagated.head(3))

### Verify Pruned Labels and Prune Them If Needed. Keep Only the Terms with no Children

In [ ]:
# Determine leaf-only labels for each protein
not_pruned_count = 0
pruned_rows = []
for prot, group in train_terms_df.groupby("EntryID"):
    terms = set(group['term'])
    pruned = terms.copy()
    for t in list(terms):
        descent_stack = []
        visited_terms = []
        descent_stack.extend(list(ont_graph.predecessors(t)))
        while descent_stack:
            child = descent_stack.pop()
            if child in terms:
                pruned.discard(t)
                break
            else:
                descent_stack.extend(list(ont_graph.predecessors(child)))
    if pruned != terms:
        not_pruned_count+=1
    pruned_rows.append(
        {
            "EntryID": prot,
            "term": list(pruned)
        }
    )

print(f"Number of labels not pruned: {not_pruned_count}")

In [ ]:
train_terms_df_pruned = pd.DataFrame(pruned_rows).explode("term")
train_terms_df_pruned['aspect'] = train_terms_df_pruned['term']\
    .map(term_ns_map)\
    .map(
        {
            'biological_process': "P",
            'cellular_component': "C",
            'molecular_function': "F"
        }
    )

print(train_terms_df_pruned.shape)
print(f"aspects per records distribution: {train_terms_df_pruned['aspect'].map(len).value_counts()}")
display(train_terms_df_pruned.head(3))

## Plots

### Distribution of Term Count per Protein

In [ ]:
fig, axs = plt.subplots(1,3, figsize = (18,4), sharex=True)

labels_per_protein = train_terms_df.groupby("EntryID")["term"].count()
axs[0].hist(labels_per_protein, bins=range(1, labels_per_protein.max()+2), edgecolor='black')
axs[0].axvline(labels_per_protein.median(), label = "Median", color = "orange")
axs[0].axvline(labels_per_protein.quantile(0.95), label = "95-Percentile", color = "magenta")
axs[0].set_xlabel("Number of GO terms annotated per protein")
axs[0].set_ylabel("Number of proteins")
axs[0].set_title("Distribution of GO terms per protein [Original]")
axs[0].legend(loc="upper right")

labels_per_protein_prop = train_terms_df_propagated.groupby("EntryID")["term"].count()
axs[1].hist(labels_per_protein_prop, bins=range(1, labels_per_protein_prop.max()+2), edgecolor='black')
axs[1].axvline(labels_per_protein_prop.median(), label = "Median", color = "orange")
axs[1].axvline(labels_per_protein_prop.quantile(0.95), label = "95-Percentile", color = "magenta")
axs[1].set_xlabel("Number of GO terms annotated per protein")
axs[1].set_ylabel("Number of proteins")
axs[1].set_title("Distribution of GO terms per protein [Propagated]")
axs[1].legend(loc="upper left")

labels_per_protein_prune = train_terms_df_pruned.groupby("EntryID")["term"].count()
axs[2].hist(labels_per_protein_prune, bins=range(1, labels_per_protein_prune.max()+2), edgecolor='black')
axs[2].axvline(labels_per_protein_prune.median(), label = "Median", color = "orange")
axs[2].axvline(labels_per_protein_prune.quantile(0.95), label = "95-Percentile", color = "magenta")
axs[2].set_xlabel("Number of GO terms annotated per protein")
axs[2].set_ylabel("Number of proteins")
axs[2].set_title("Distribution of GO terms per protein [Pruned]")
axs[2].legend(loc="upper right")


plt.xscale('log')
plt.show()

### Distribution of Term Count per Namespace & Depth

In [ ]:
fig, axs = plt.subplots(1,3, figsize = (18,4), sharex=True, sharey=True)


df_dict =  {
    'Original': train_terms_df,
    'Propagated': train_terms_df_propagated,
    'Pruned': train_terms_df_pruned
}

for i, (label, df) in enumerate(df_dict.items()):
    df["depth"] = df["term"].map(depths)
    axs[i].hist(
        [
            df.loc[df['aspect'] == "P", "depth"],
            df.loc[df['aspect'] == "F", "depth"],
            df.loc[df['aspect'] == "C", "depth"]
        ],
        label=["BP", "MF", "CC"], color=["orange","gold","salmon"],
        stacked=False, alpha=0.7, bins=range(0, 15),
    )
    axs[i].set_xlabel("Depth in GO hierarchy")
    axs[i].set_ylabel("Number of GO terms (in training set)")
    axs[i].set_title(f"GO term count by depth and sub-ontology [{label}]")
    axs[i].legend()

plt.show()

### Distribution of Records Counts per GO Term

In [ ]:
fig, axs = plt.subplots(1,3, figsize = (18,4), sharex=False)

train_terms_df["term"].value_counts().plot(kind='hist', bins=200, label = 'Original', ax = axs[0])
train_terms_df_propagated["term"].value_counts().plot(kind='hist', bins=200, label = 'Propagated', ax = axs[1])
train_terms_df_pruned["term"].value_counts().plot(kind='hist', bins=200, label = 'Pruned', ax = axs[2])

for ax in axs:
    ax.set_title("Distribution of Records Counts per GO Term")
    ax.legend()
    ax.set_yscale('log')

### Distribution of AA terms

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

test_aa_list = [list(set(prot.seq)) for prot in test_fasta_sequences]
train_aa_list = [list(set(prot.seq)) for prot in train_fasta_sequences]

test_aa_mlb = MultiLabelBinarizer()
train_aa_mlb = MultiLabelBinarizer()
test_binary_aa = test_aa_mlb.fit_transform(test_aa_list)
train_binary_aa = train_aa_mlb.fit_transform(train_aa_list)

# print(f"Test AA elements: {test_aa_mlb.classes_} | length: {len(test_aa_mlb.classes_)}")
# print(f"Train AA elements: {train_aa_mlb.classes_} | length: {len(train_aa_mlb.classes_)}")

fig, axs = plt.subplots(1,2, figsize = (12,5), sharex = True)

axs[0].bar(test_aa_mlb.classes_, test_binary_aa.sum(axis=0))
axs[0].set_title("Distribution of Amino Acid Elements [Test Data]")
axs[0].set_xlabel("AA Elements")
axs[0].set_ylabel("Number of Proteins Containing the Element")
axs[1].bar(train_aa_mlb.classes_, train_binary_aa.sum(axis=0))
axs[1].set_title("Distribution of Amino Acid Elements [Train Data]")
axs[1].set_xlabel("AA Elements")
axs[1].set_ylabel("Number of Proteins Containing the Element")

# TODO: we might need to remove proteins, including the "X" element, if
    # it's needed to calculate BLAST/DIAMOND
    # or the PLM (protein LM) requires it as a prep step

### Distribution of Sequence Lengths

In [ ]:
test_seq_len_list = [len(prot.seq) for prot in test_fasta_sequences]
train_seq_len_list = [len(prot.seq) for prot in train_fasta_sequences]

fig, axs = plt.subplots(1,2, figsize = (12,5), sharex = True)

axs[0].hist(test_seq_len_list, bins = 100)
axs[0].axvline(np.quantile(test_seq_len_list, 0.98), label = "98% quantile", color="red")
axs[0].text(0.15, 0.9, f"{np.quantile(test_seq_len_list, 0.98)}",transform=axs[0].transAxes)
axs[0].set_title("Distribution of Sequence Length [Test Data]")
axs[0].set_xlabel("Sequence Length")
axs[0].set_ylabel("Freq")
axs[1].hist(train_seq_len_list, bins = 100)
axs[1].axvline(np.quantile(train_seq_len_list, 0.98), label = "98% quantile", color="red")
axs[1].text(0.15, 0.9, f"{np.quantile(train_seq_len_list, 0.98)}",transform=axs[1].transAxes)
axs[1].set_title("Distribution of Sequence Length [Train Data]")
axs[1].set_xlabel("Sequence Length")
axs[1].set_ylabel("Freq")

for ax in axs:
    ax.set_yscale('log')
    ax.legend()


# TODO: Search for any sequence length limitation for BLAST/DIAMOND or the PLM used.
    # If needed, choose a max seq length, pad smaller sequences, and ignore longer ones.